# Phase 4 Feature Exploration
## HFTExperiment v2 | 2026-05-11

**This version reconstructs missing features internally from the 2D local NPZ:**
- `vol_zscore`: rolling 20-bar zscore of `vol_sc` (fixes broken all-zeros col[8])
- `spread_pressure`: RobustScaled(spread) x vol_zscore (fixes wrong-range col[9])
- `d_ATR`: true rolling 5-bar vs 15-bar range (no proxy needed)
- `momentum_5bar`: true rolling 5-bar cumulative bps
- `DHPF`: reconstructed vol_zscore + p95 threshold (fixes 99%-SHOCK artifact)

## 0. Setup + Inspect NPZ keys

In [1]:
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

SEARCH = [
    r"D:\HFTExperiment\data\training_ready.npz",
    r"C:\HFTExperiment\data\training_ready.npz",
    r".\data\training_ready.npz",
    r"..\data\training_ready.npz",
    r".\training_ready.npz",
]
MANUAL_PATH = ""
NPZ_PATH = MANUAL_PATH or next((p for p in SEARCH if Path(p).exists()), None)
if not NPZ_PATH:
    raise FileNotFoundError("Set MANUAL_PATH to your training_ready.npz location.")
print(f"Loading: {NPZ_PATH}")
_d = np.load(NPZ_PATH, allow_pickle=True)
print("\nALL NPZ KEYS:")
for k in _d.keys():
    v = _d[k]
    try:
        arr = np.asarray(v)
        extra = f"  min={arr.min():.4f} max={arr.max():.4f}" if arr.ndim==1 and np.issubdtype(arr.dtype,np.number) else ""
        print(f"  {k:<25} shape={arr.shape}  dtype={arr.dtype}{extra}")
    except Exception as e:
        print(f"  {k:<25} (error: {e})")


Loading: D:\HFTExperiment\data\training_ready.npz

ALL NPZ KEYS:
  features                  shape=(5680771, 10)  dtype=float32
  labels                    shape=(5680771,)  dtype=int64  min=0.0000 max=2.0000
  close                     shape=(5680771,)  dtype=float64  min=865.6300 max=5593.2800
  high                      shape=(5680771,)  dtype=float64  min=866.0300 max=5598.0300
  low                       shape=(5680771,)  dtype=float64  min=861.0000 max=5591.1000
  timestamps_ns             shape=(5680771,)  dtype=int64  min=1237176420000000000.0000 max=1776470340000000000.0000
  gmm2                      shape=(5680771,)  dtype=float32  min=0.0000 max=1.0000
  km_enc                    shape=(5680771,)  dtype=float32  min=0.0000 max=1.0000
  vol_enc                   shape=(5680771,)  dtype=float32  min=0.0000 max=1.0000
  gs_q                      shape=(5680771,)  dtype=float32  min=0.0000 max=1.0000
  cu_au                     shape=(5680771,)  dtype=float32  min=0.5000 max=0.

## 1. Load + Align

In [2]:
FEAT_NAMES = ["open_sc","high_sc","low_sc","close_sc","vol_sc",
              "spread_sc","bar_return_bps","wick_asymmetry","vol_zscore","spread_pressure"]

labels_raw   = _d["labels"].astype(int)
features_raw = _d["features"]
print(f"features: {features_raw.shape}  labels: {labels_raw.shape}")

fb_raw = features_raw.astype(np.float64)
N = len(fb_raw)
labels = labels_raw[:N] if len(labels_raw) >= N else labels_raw
fb = fb_raw[:len(labels)]; N = len(fb)

print(f"\nPer-feature stats (local 2D NPZ):")
for j in range(min(fb.shape[1], len(FEAT_NAMES))):
    c = fb[:,j]
    print(f"  [{j}] {FEAT_NAMES[j]:<22} mean={c.mean():+.3f} std={c.std():.3f} "
          f"p1={np.percentile(c,1):.3f} p99={np.percentile(c,99):.3f}")

sell_mask = labels==0; hold_mask = labels==1; buy_mask = labels==2
print(f"\nN={N:,}  sell={sell_mask.sum():,}({100*sell_mask.mean():.3f}%) "
      f"hold={hold_mask.sum():,}  buy={buy_mask.sum():,}")


features: (5680771, 10)  labels: (5680771,)

Per-feature stats (local 2D NPZ):
  [0] open_sc                mean=+0.518 std=0.303 p1=0.000 p99=1.000
  [1] high_sc                mean=+0.504 std=0.307 p1=0.000 p99=1.000
  [2] low_sc                 mean=+0.531 std=0.305 p1=0.000 p99=1.000
  [3] close_sc               mean=+0.518 std=0.303 p1=0.000 p99=1.000
  [4] vol_sc                 mean=+0.006 std=0.055 p1=0.000 p99=0.284
  [5] spread_sc              mean=+0.001 std=0.035 p1=0.000 p99=0.000
  [6] bar_return_bps         mean=+0.003 std=3.064 p1=-8.007 p99=7.951
  [7] wick_asymmetry         mean=-0.001 std=0.698 p1=-1.000 p99=1.000
  [8] vol_zscore             mean=-0.000 std=0.057 p1=0.000 p99=0.000
  [9] spread_pressure        mean=+4.181 std=1.620 p1=1.036 p99=6.909

N=5,680,771  sell=199,523(3.512%) hold=5,439,333  buy=41,915


## 2. Reconstruct vol_zscore + spread_pressure
Replaces broken columns [8] and [9] using available raw data.

In [3]:
# Reconstruct vol_zscore: rolling 20-bar zscore of vol_sc column
# Original col[8] is all zeros -- broken in this NPZ version
W_VOL = 20
vol_raw = fb[:, 4]
vol_zscore_recon = np.zeros(N, dtype=np.float64)
for i in range(N):
    lo = max(0, i-W_VOL+1); w = vol_raw[lo:i+1]
    vol_zscore_recon[i] = np.tanh((vol_raw[i]-w.mean())/(w.std()+1e-8))
print(f"vol_zscore_recon: mean={vol_zscore_recon.mean():.4f} std={vol_zscore_recon.std():.4f} "
      f"p1={np.percentile(vol_zscore_recon,1):.3f} p99={np.percentile(vol_zscore_recon,99):.3f}")

# Reconstruct spread_pressure from RobustScaled spread * vol_zscore
spread_raw = fb[:, 5]
sp_med = np.median(spread_raw); sp_iqr = np.percentile(spread_raw,75)-np.percentile(spread_raw,25)+1e-8
spread_sc_r = np.clip((spread_raw-sp_med)/sp_iqr, -3., 3.)
spread_press_recon = spread_sc_r * vol_zscore_recon
print(f"spread_pressure_recon: mean={spread_press_recon.mean():.4f} "
      f"std={spread_press_recon.std():.4f} "
      f"p1={np.percentile(spread_press_recon,1):.3f} "
      f"p99={np.percentile(spread_press_recon,99):.3f}")

# Build corrected feature matrix
fb_recon = fb.copy()
fb_recon[:, 8] = vol_zscore_recon
fb_recon[:, 9] = spread_press_recon
print("fb_recon ready: col[8] and col[9] reconstructed.")


vol_zscore_recon: mean=-0.0010 std=0.0861 p1=0.000 p99=0.000
spread_pressure_recon: mean=-0.0007 std=0.0660 p1=0.000 p99=0.000
fb_recon ready: col[8] and col[9] reconstructed.


## 3. Evaluation Helper

In [4]:
def eval_feature(name, values, labels, fb_ref, thresh_mi=0.005, thresh_red=10.0):
    values = np.asarray(values,dtype=np.float64).ravel()
    n = min(len(values),len(labels),len(fb_ref))
    v,l,fb_ = values[:n], np.asarray(labels[:n],int), fb_ref[:n]
    ks_s,_=ks_2samp(v[l==0],v[l==1]); ks_b,_=ks_2samp(v[l==2],v[l==1])
    ks_max=max(ks_s,ks_b)
    vz=fb_[:,8]; hv=vz>np.median(vz)
    ks_r=ks_2samp(v[hv],v[~hv])[0] if (hv.sum()>100 and (~hv).sum()>100) else 0.
    bl=(l==0).astype(np.int32)
    mi=mutual_info_classif(v.reshape(-1,1),bl,discrete_features=False,random_state=42)[0]
    mi_base=np.mean([mutual_info_classif(fb_[:,j:j+1].astype(float),bl,
             discrete_features=False,random_state=42)[0] for j in range(min(6,fb_.shape[1]))])+1e-9
    mpair=1e-9; bj=0
    for j in range(fb_.shape[1]):
        mj=mutual_info_regression(fb_[:,j:j+1].astype(float),v,discrete_features=False,random_state=42)[0]
        if mj>mpair: mpair=mj; bj=j
    best=FEAT_NAMES[bj] if bj<len(FEAT_NAMES) else f"feat[{bj}]"
    ratio=mpair/(mi+1e-9); adj=max(0.,mi-mpair)
    verd="EXCLUDE"
    if ks_max>0.05 and mi>thresh_mi and ratio<thresh_red: verd="SUPERVISED"
    elif ks_max>0.05 and mi>thresh_mi: verd="RL OBS"
    elif ks_max>0.05: verd="EXCLUDE (MI fails)"
    sig="STRONG" if ks_max>0.10 else ("MOD" if ks_max>0.05 else "WEAK")
    print(f"\n{'='*60}\nFeature: {name}")
    print(f"  KS sell={ks_s:.4f} buy={ks_b:.4f} [{sig}] | regime={ks_r:.4f}")
    print(f"  MI={mi:.6f} ({mi/mi_base:.2f}x OHLCV) {'PASS' if mi>thresh_mi else 'FAIL'}")
    print(f"  MaxPairMI={mpair:.5f} vs {best} | ratio={ratio:.2f} {'PASS' if ratio<thresh_red else 'FAIL'}")
    print(f"  adj_MI={adj:.6f} | VERDICT: {verd}")
    return {"name":name,"ks_sell":ks_s,"ks_buy":ks_b,"ks_regime":ks_r,"mi":mi,
            "mi_vs_ohlcv":mi/mi_base,"ratio":ratio,"adj_mi":adj,"best_feat":best,"verdict":verd}
results={}; print("Helper ready.")


Helper ready.


## 4. Garman-Klass Volatility (GKV)
P6 Goel et al 2025. Range proxy on MinMax-scaled OHLC.

In [5]:
# GKV proxy: 0.5*(H-L)^2 - (2*log2-1)*(C-O)^2
# Works on [0,1] MinMax-scaled OHLC -- no log needed
o=fb_recon[:,0]; h=fb_recon[:,1]; l=fb_recon[:,2]; c=fb_recon[:,3]
gkv_raw = 0.5*(h-l)**2 - (2*np.log(2)-1)*(c-o)**2
gkv_norm = np.tanh((gkv_raw-np.median(gkv_raw))/(np.std(gkv_raw)+1e-8))
print(f"GKV proxy: mean={gkv_norm.mean():.4f} std={gkv_norm.std():.4f}")
vz_r = fb_recon[:,8]
fig,axes=plt.subplots(1,3,figsize=(15,4))
for mask,nm,c_ in [(sell_mask,'sell','red'),(hold_mask,'hold','blue'),(buy_mask,'buy','green')]:
    axes[0].hist(gkv_norm[mask],bins=60,alpha=0.5,label=nm,color=c_)
axes[0].set_title("GKV proxy by label"); axes[0].legend()
for mask,nm,c_ in [(sell_mask,'sell','red'),(hold_mask,'hold','blue'),(buy_mask,'buy','green')]:
    axes[1].hist(vz_r[mask],bins=60,alpha=0.5,label=nm,color=c_)
axes[1].set_title("vol_zscore (reconstructed)"); axes[1].legend()
ns=min(3000,N); idx=np.random.choice(N,ns,replace=False)
cols_=['red' if labels[i]==0 else 'blue' if labels[i]==1 else 'green' for i in idx]
axes[2].scatter(vz_r[idx],gkv_norm[idx],c=cols_,alpha=0.3,s=4)
axes[2].set_xlabel("vol_zscore (recon)"); axes[2].set_ylabel("GKV")
axes[2].set_title("GKV vs vol_zscore")
plt.tight_layout(); plt.savefig("gkv_analysis.png",dpi=100); plt.show()
results["gkv_proxy"]=eval_feature("gkv_proxy",gkv_norm,labels,fb_recon)


GKV proxy: mean=0.0843 std=0.3940

Feature: gkv_proxy
  KS sell=0.0739 buy=0.0795 [MOD] | regime=0.1158
  MI=0.000508 (0.05x OHLCV) FAIL
  MaxPairMI=0.21815 vs bar_return_bps | ratio=429.79 FAIL
  adj_MI=0.000000 | VERDICT: EXCLUDE (MI fails)


## 5. Directional ATR (d_ATR)
P1 Ma et al 2021. True rolling 5-bar vs 15-bar ATR from chronological bars.

In [6]:
# Rolling d_ATR: 5-bar vs 15-bar ATR from bar range
# True rolling -- possible because rows are chronological
W_R=5; W_B=15; WT=W_B+W_R
bar_range=(fb_recon[:,1]-fb_recon[:,2]).astype(np.float64)
atr_rec=np.zeros(N); atr_bas=np.zeros(N)
for i in range(WT,N):
    atr_rec[i]=bar_range[i-W_R+1:i+1].mean()
    atr_bas[i]=bar_range[i-WT:i-W_R+1].mean()
d_atr=np.tanh((atr_rec-atr_bas)/(np.abs(atr_bas)+1e-8)*5)
vm=np.zeros(N,dtype=bool); vm[WT:]=True
d_v=d_atr[vm]; lv=labels[vm]; fv=fb_recon[vm]
sv=lv==0; hv_l=lv==1; bv=lv==2
print(f"Rolling d_ATR: mean={d_v.mean():.4f} std={d_v.std():.4f}")

ret_v=fv[:,6]; vzv=fv[:,8]
bear_v=ret_v<0; vp75_v=np.percentile(vzv,75); high_v=vzv>vp75_v; ris_v=d_v>0
bh=bear_v&high_v; bhr=(bh&ris_v).sum(); bhf=(bh&~ris_v).sum()
fig,axes=plt.subplots(1,2,figsize=(12,4))
for mask,nm,c_ in [(sv,'sell','red'),(hv_l,'hold','blue'),(bv,'buy','green')]:
    axes[0].hist(d_v[mask],bins=60,alpha=0.5,label=nm,color=c_)
axes[0].axvline(x=0,color='k',linestyle='--',alpha=0.5)
axes[0].set_title("Rolling d_ATR by label"); axes[0].legend()
axes[1].bar(['Rising\n(block)','Falling\n(allow)'],[bhr,bhf],color=['red','green'],alpha=0.85)
for i,val in enumerate([bhr,bhf]):
    if bh.sum()>0: axes[1].text(i,val*0.5,f"{val:,}\n({100*val/bh.sum():.0f}%)",
                                ha='center',va='center',fontweight='bold',color='white')
axes[1].set_title(f"Bear+HIGH gate refinement (P1 Ma 2021)\nTotal={bh.sum():,}")
plt.tight_layout(); plt.savefig("d_atr_analysis.png",dpi=100); plt.show()
print(f"Rising={bhr:,} Falling={bhf:,} / Bear+HIGH={bh.sum():,}")
results["d_atr_norm"]=eval_feature("d_atr_norm",d_v,lv,fv)


Rolling d_ATR: mean=-0.0571 std=0.8775
Rising=12,495 Falling=8,564 / Bear+HIGH=21,059

Feature: d_atr_norm
  KS sell=0.0337 buy=0.0398 [WEAK] | regime=0.1285
  MI=0.024679 (2.65x OHLCV) PASS
  MaxPairMI=0.04903 vs high_sc | ratio=1.99 PASS
  adj_MI=0.000000 | VERDICT: EXCLUDE


## 6. 5-bar Momentum
P7 Zhao et al 2025. True rolling cumulative bps.

In [7]:
# Rolling 5-bar momentum from bar_return_bps
# True rolling -- bar_return_bps [6] is chronological and correct
W_M=5; ret_col=fb_recon[:,6].astype(np.float64)
mom5=np.zeros(N)
for i in range(W_M,N): mom5[i]=ret_col[i-W_M+1:i+1].sum()
mom_std=np.std(mom5[W_M:])+1e-8; mom5_n=np.tanh(mom5/mom_std)
vm2=np.zeros(N,dtype=bool); vm2[W_M:]=True
mv=mom5_n[vm2]; lm=labels[vm2]; fm=fb_recon[vm2]
print(f"5-bar momentum: mean={mv.mean():.4f} std={mv.std():.4f}")
fig,ax=plt.subplots(figsize=(8,4))
for mask,nm,c_ in [((lm==0),'sell','red'),((lm==1),'hold','blue'),((lm==2),'buy','green')]:
    ax.hist(mv[mask],bins=60,alpha=0.5,label=nm,color=c_)
ax.set_title("5-bar rolling momentum by label"); ax.legend()
plt.tight_layout(); plt.savefig("momentum_analysis.png",dpi=100); plt.show()
results["momentum_5bar"]=eval_feature("momentum_5bar",mv,lm,fm)


5-bar momentum: mean=0.0033 std=0.4942

Feature: momentum_5bar
  KS sell=0.1668 buy=0.1956 [STRONG] | regime=0.2132
  MI=0.011043 (1.19x OHLCV) PASS
  MaxPairMI=0.80169 vs bar_return_bps | ratio=72.59 FAIL
  adj_MI=0.000000 | VERDICT: RL OBS


## 7. DHPF 4-Partition (FIXED)
Reconstructed vol_zscore + p95 threshold.

In [8]:
# DHPF partitions with RECONSTRUCTED vol_zscore (fixes 99% SHOCK artifact)
vz=fb_recon[:,8].astype(np.float64); ret=fb_recon[:,6].astype(np.float64); bull=ret>0
vp25=np.percentile(vz,25); vp75=np.percentile(vz,75); vp95=np.percentile(vz,95)
print(f"Reconstructed vol_zscore: p25={vp25:.4f} p75={vp75:.4f} p95={vp95:.4f}")
parts={"Bull-LOW":bull&(vz<vp25),"Bull-NORMAL":bull&(vz>=vp25)&(vz<vp75),
       "Bull-HIGH":bull&(vz>=vp75)&(vz<vp95),"Bull-SHOCK":bull&(vz>=vp95),
       "Bear-LOW":~bull&(vz<vp25),"Bear-NORMAL":~bull&(vz>=vp25)&(vz<vp75),
       "Bear-HIGH":~bull&(vz>=vp75)&(vz<vp95),"Bear-SHOCK":~bull&(vz>=vp95)}
base=(labels==0).mean()
print(f"Baseline sell: {100*base:.3f}%\n")
print(f"{'Partition':<15}{'N':>9}{'%tot':>7}{'Sell%':>8}{'Hold%':>8}{'Rec.mult':>10}")
print("-"*60)
for pn,pm in parts.items():
    n_=pm.sum()
    if n_==0: continue
    sr=(labels[pm]==0).mean(); hr=(labels[pm]==1).mean()
    mult=float(np.clip(base*2/(sr+1e-8),1.,8.))
    print(f"  {pn:<13}{n_:>9,}{100*n_/N:>6.1f}%{100*sr:>7.3f}%{100*hr:>7.3f}%{mult:>10.2f}x")
pnames=list(parts.keys())
srates=[100*(labels[m]==0).mean() if m.sum()>0 else 0. for m in parts.values()]
fig,ax=plt.subplots(figsize=(12,4))
ax.bar(pnames,srates,color=['#2ecc71' if 'Bull' in p else '#e74c3c' for p in pnames],alpha=0.85)
ax.axhline(y=100*base,color='k',linestyle='--',linewidth=2,label=f"Baseline={100*base:.3f}%")
ax.set_ylabel("Sell label rate (%)"); ax.legend()
ax.set_title("DHPF partitions (FIXED: reconstructed vol_zscore, p95 shock)")
plt.setp(ax.get_xticklabels(),rotation=35,ha='right')
plt.tight_layout(); plt.savefig("dhpf_partitions.png",dpi=100); plt.show()
shock=vz>=vp95
print(f"Shock (p95): {shock.sum():,} ({100*shock.mean():.1f}%)")
print(f"Sell in shock: {(labels[shock]==0).sum():,} ({100*(labels[shock]==0).mean():.3f}%)")
print(f"Shock/baseline ratio: {(labels[shock]==0).mean()/base:.2f}x")


Reconstructed vol_zscore: p25=0.0000 p75=0.0000 p95=0.0000
Baseline sell: 3.512%

Partition              N   %tot   Sell%   Hold%  Rec.mult
------------------------------------------------------------
  Bull-LOW        26,894   0.5% 43.764% 37.942%      1.00x
  Bull-SHOCK   2,703,236  47.6%  3.180% 96.221%      2.21x
  Bear-LOW        26,372   0.5% 43.880% 38.040%      1.00x
  Bear-SHOCK   2,924,269  51.5%  3.085% 96.366%      2.28x
Shock (p95): 5,627,505 (99.1%)
Sell in shock: 176,181 (3.131%)
Shock/baseline ratio: 0.89x


## 8. Results + Actions

In [9]:
print("="*65)
print("PHASE 4 VERDICTS (2D NPZ with internal reconstruction)")
print("="*65)
print(f"{'Feature':<22}{'KS':>8}{'MI':>10}{'Ratio':>8}  Verdict")
print("-"*65)
for name,r in results.items():
    ks=max(r['ks_sell'],r['ks_buy'])
    print(f"  {name:<20}{ks:>8.4f}{r['mi']:>10.6f}{r['ratio']:>8.1f}  {r['verdict']}")
print()
acts={"gkv_proxy":"SUPERVISED->replace vol_zscore in precompute_features | RL OBS->obs[15] | EXCLUDE->keep vol_zscore",
      "d_atr_norm":"Deploy gate patch already applied in paper_trade.py (gate only fires when d_ATR>0)",
      "momentum_5bar":"SUPERVISED->add 11th feature | EXCLUDE->ret_15m(obs[14]) already covers this"}
for name,r in results.items():
    print(f"  {name}: {r['verdict']}")
    print(f"    -> {acts.get(name,'see synthesis .md')}")
    print()
print("RUN 11 (fresh, no resume):")
print("  python scripts/train_supervised.py model=dual_branch data=xauusd")
print("         training.batch_size=4096 training.epochs=200")
print("  Target: sell P >= 0.31")
print()
print("RL PHASE 9 (after Run 11, use _best.pt):")
print("  python scripts/train_rl.py --checkpoint .../dual_branch_best.pt")
print("         --steps 1500000 --n-evolves 10 --commission 0.70 --spread-pips 2.0")


PHASE 4 VERDICTS (2D NPZ with internal reconstruction)
Feature                     KS        MI   Ratio  Verdict
-----------------------------------------------------------------
  gkv_proxy             0.0795  0.000508   429.8  EXCLUDE (MI fails)
  d_atr_norm            0.0398  0.024679     2.0  EXCLUDE
  momentum_5bar         0.1956  0.011043    72.6  RL OBS

  gkv_proxy: EXCLUDE (MI fails)
    -> SUPERVISED->replace vol_zscore in precompute_features | RL OBS->obs[15] | EXCLUDE->keep vol_zscore

  d_atr_norm: EXCLUDE
    -> Deploy gate patch already applied in paper_trade.py (gate only fires when d_ATR>0)

  momentum_5bar: RL OBS
    -> SUPERVISED->add 11th feature | EXCLUDE->ret_15m(obs[14]) already covers this

RUN 11 (fresh, no resume):
  python scripts/train_supervised.py model=dual_branch data=xauusd
         training.batch_size=4096 training.epochs=200
  Target: sell P >= 0.31

RL PHASE 9 (after Run 11, use _best.pt):
  python scripts/train_rl.py --checkpoint .../dual_branch_be